In [2]:
import scanpy as sc
import pickle
import os
import anndata
import mudata
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import ArrowStyle
from scipy.stats import fisher_exact
import seaborn as sns
import glob
import PyComplexHeatmap as pch
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib_venn import venn2
from ast import literal_eval
from scipy.stats import hypergeom
from scipy.stats import pearsonr
import numpy as np
from scipy.stats import pearsonr, spearmanr
import warnings
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import SpectralCoclustering,SpectralBiclustering
from sklearn.metrics import consensus_score
from statsmodels.stats.multitest import multipletests
import glob as glob


In [20]:
outdir="/data2st2/junyi/output/sn0615"

# Read the meta data 

In [5]:
if os.path.exists("/data2st2/junyi/output/stg1028/combined_ALL_meta.csv"):
    df_meta = pd.read_csv("/data2st2/junyi/output/stg1028/combined_ALL_meta.csv", index_col=0)
    df_meta.drop_duplicates(inplace=True)
else:
    df_meta = pd.DataFrame()
    h5ads = glob.glob('/data2st2/junyi/output/stg1028/*_4VN/*.h5ad')
    for h5ad in h5ads:
        adata = sc.read_h5ad(h5ad, backed='r')
        df_meta = pd.concat([df_meta, pd.DataFrame(adata.obs)], axis=0)
    df_meta.to_csv("/data2st2/junyi/output/stg1028/combined_ALL_meta.csv")
    df_meta.drop_duplicates(inplace=True)

/tmp/ipykernel_3091102/3001891177.py:2: DtypeWarning: Columns (20,56,57,58,67,68) have mixed types. Specify dtype option on import or set low_memory=False.
  df_meta = pd.read_csv("/data2st2/junyi/output/stg1028/combined_ALL_meta.csv", index_col=0)


In [26]:
# Read the meta data and create a map between desired maps
df_meta['Region Subclass'] = df_meta['region'] + "_" + df_meta['celltype.L2']
df_meta['ctname'] = df_meta['Region Subclass'].str.replace(" ","_")
df_meta['ctname'] = df_meta['ctname'].str.replace("/","_")
df_meta['bacode'] = df_meta.index.str[:18]

In [13]:
df_meta.status.value_counts()

status
SUS      557716
CON      540042
CSRES    291287
RES      285130
CSDS     214690
Name: count, dtype: int64

In [33]:
df_meta_M3R = df_meta[df_meta['region'].isin(['AMY', 'HPF', 'PFC'])].copy()
df_meta_M3R = df_meta_M3R[df_meta_M3R['status'].isin(['SUS', 'CON'])]
df_meta_M3R = df_meta_M3R[df_meta_M3R.sex == 'M']

In [48]:
df_selected_sample = df_meta_M3R.drop_duplicates(subset=['sample'])

In [50]:
df_selected_sample.to_csv(os.path.join(outdir, "selected_samples.csv"), index=False)

In [36]:
# 为每个 sample 创建文件夹，并为每个 ctname 生成 barcode 表
import os

# 使用 df_meta_M3R (AMY/HIP/PFC, MALE, SUS/CON)
# 为每个 sample 创建文件夹
sample_list = df_meta_M3R['sample'].unique()
print(f"样本数: {len(sample_list)}")
print(f"样本列表: {sample_list}")

# 在 outdir 下创建 sample 文件夹
for sample in sample_list:
    sample_dir = os.path.join(outdir, "sample_barcodes", sample)
    os.makedirs(sample_dir, exist_ok=True)
    
    # 获取该 sample 的所有 cell
    sample_df = df_meta_M3R[df_meta_M3R['sample'] == sample]
    
    # 为该 sample 中的每个 ctname 生成一个 barcode 表
    for ctname, ct_df in sample_df.groupby('ctname'):
        # 提取 barcode，每一行一个 barcode
        barcodes = ct_df[['bacode']].drop_duplicates()
        outfile = os.path.join(sample_dir, f"{ctname}_barcodes.csv")
        barcodes.to_csv(outfile, header=False, index=False)
    
    print(f"  {sample}: {sample_df['ctname'].nunique()} 个 celltype")

print("\n完成！每个 sample 文件夹下包含各 ctname 的 barcode 表。")


样本数: 24
样本列表: ['MW26B_PFC_novogene' 'MW26E_PFC_yunzhun' 'MW45A_AMY_yunzhun'
 'MW45C_AMY_beirui' 'MW45C_HPF_beirui' 'MW26E_AMY_yunzhun'
 'MW51A_HPF_yunzhun' 'MW22B_AMY_yunzhun' 'MW45A_HPF_yunzhun'
 'MW22B_HPF_yunzhun' 'MW34C_PFC_beirui' 'MW51A_PFC_yunzhun'
 'MC33A_HPF_yunzhun' 'MC37A_AMY_yunzhun' 'MC25B_PFC_yunzhun'
 'MC48E_HPF_beirui' 'MC21D_PFC_beirui' 'MC33A_PFC_novogene'
 'MC48E_AMY_beirui' 'MC48D_HPF_yunzhun' 'MC52E_PFC_yunzhun'
 'MC25B_HPF_yunzhun' 'MC25B_AMY_yunzhun' 'MC48D_AMY_yunzhun']
  MW26B_PFC_novogene: 38 个 celltype
  MW26E_PFC_yunzhun: 35 个 celltype
  MW45A_AMY_yunzhun: 40 个 celltype
  MW45C_AMY_beirui: 39 个 celltype
  MW45C_HPF_beirui: 36 个 celltype
  MW26E_AMY_yunzhun: 40 个 celltype
  MW51A_HPF_yunzhun: 35 个 celltype
  MW22B_AMY_yunzhun: 37 个 celltype
  MW45A_HPF_yunzhun: 41 个 celltype
  MW22B_HPF_yunzhun: 35 个 celltype
  MW34C_PFC_beirui: 34 个 celltype
  MW51A_PFC_yunzhun: 34 个 celltype
  MC33A_HPF_yunzhun: 40 个 celltype
  MC37A_AMY_yunzhun: 44 个 celltype
  MC25B_PFC_y

In [41]:
bamfiles = glob.glob("/data1st2/mark/snRNA/*/cellranger_output*/*/cellranger_output/*/outs/possorted_genome_bam.bam")

In [ ]:
#之找出

['/data1st2/mark/snRNA/snRNA_CUMS/cellranger_output_raw/snRNA_batch_novogene_out/cellranger_output/MC33A_PFC_novogene/outs/possorted_genome_bam.bam',
 '/data1st2/mark/snRNA/snRNA_CUMS/cellranger_output_raw/snRNA_batch_novogene_out/cellranger_output/MW26B_PFC_novogene/outs/possorted_genome_bam.bam',
 '/data1st2/mark/snRNA/snRNA_CUMS/cellranger_output_raw/snRNA_batch_beirui_out/cellranger_output/FW55B_TH_beirui/outs/possorted_genome_bam.bam',
 '/data1st2/mark/snRNA/snRNA_CUMS/cellranger_output_raw/snRNA_batch_beirui_out/cellranger_output/MW45A_HY_beirui/outs/possorted_genome_bam.bam',
 '/data1st2/mark/snRNA/snRNA_CUMS/cellranger_output_raw/snRNA_batch_beirui_out/cellranger_output/MC52E_MB_beirui/outs/possorted_genome_bam.bam',
 '/data1st2/mark/snRNA/snRNA_CUMS/cellranger_output_raw/snRNA_batch_beirui_out/cellranger_output/MW51A_STR_beirui/outs/possorted_genome_bam.bam',
 '/data1st2/mark/snRNA/snRNA_CUMS/cellranger_output_raw/snRNA_batch_beirui_out/cellranger_output/FC56E_HY_beirui/outs/p